# Module 7 Learning Activities: Model Evaluation with PySpark

This notebook is a beginner-friendly practical for **Learning Activity 1: Model Evaluation**.

We will build and compare two decision tree models:

- **Model 1:** predict `Income` using `Marital status` and `Cap_Gains_Losses`
- **Model 2:** predict `Income` using `Marital status` and `age`

We will evaluate each model using:

- a contingency table (confusion matrix);
- accuracy;
- precision;
- recall; and
- F1 score.

The code is intentionally kept simple and follows the same step-by-step style as the original `Session7.ipynb`.

## Learning Activity 1: Tasks

Using the `adult_ch6_training.csv` and `adult_ch6_test.csv` datasets:

1. Train **Model 1** using `Marital status` and `Cap_Gains_Losses`.
2. Evaluate Model 1 on the test data.
3. Train **Model 2** using `Marital status` and `age`.
4. Evaluate Model 2 on the test data.
5. Compare the models using accuracy, precision, recall, and F1 score.
6. Summarise your observations in the discussion forum.

> **Important:** The training data are used to learn the model. The test data must only be used for evaluation.

## 1. Import the required libraries

The libraries below provide:

- `pandas`: small summary tables;
- `SparkSession`: starts and controls Spark;
- `StringIndexer`: converts text categories into numeric values;
- `VectorAssembler`: combines input columns into one `features` vector;
- `DecisionTreeClassifier`: builds the classification model;
- `Pipeline`: applies several preparation steps in order;
- Spark evaluators: calculate performance metrics; and
- `classification_report`: displays per-class precision, recall, and F1 scores.

In [1]:
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

from sklearn.metrics import classification_report

## 2. Start a Spark session

`SparkSession.builder` creates a Spark application.

- `appName("ModelEvaluation")` gives the application a readable name.
- `getOrCreate()` creates a new session, or reuses an existing one.

In [2]:
spark = SparkSession.builder.appName("ModelEvaluation").getOrCreate()

## 3. Automatically download the training and test datasets

The two CSV files are stored in a shared Google Drive location. We will use **`gdown`** to download them directly into the current Colab or Jupyter session.

`gdown` is a small Python tool designed for downloading publicly shared Google Drive files.

In the next code cell:

- `pip install -q gdown` installs the package;
- `--fuzzy` allows `gdown` to understand a normal Google Drive sharing link;
- `-O` specifies the filename to save locally; and
- the files are saved in the current notebook working folder.

In [3]:
# Install gdown quietly
!pip install -q gdown

# Download the training dataset
!gdown --fuzzy "https://drive.google.com/file/d/1w-dEtlv25UGSFUxUXUJ6e3fGazIpDrEs/view?usp=drive_link" -O adult_ch6_training.csv

# Download the test dataset
!gdown --fuzzy "https://drive.google.com/file/d/1L5qoNIdG_Y9Jr3148ObtlHe13L1clkJ_/view?usp=drive_link" -O adult_ch6_test.csv


Downloading...
From: https://drive.google.com/uc?id=1w-dEtlv25UGSFUxUXUJ6e3fGazIpDrEs
To: /content/adult_ch6_training.csv
100% 427k/427k [00:00<00:00, 98.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1L5qoNIdG_Y9Jr3148ObtlHe13L1clkJ_
To: /content/adult_ch6_test.csv
100% 140k/140k [00:00<00:00, 75.3MB/s]


### Check that the files were downloaded

`os.path.exists()` returns `True` when a file is available in the current working folder.


In [4]:
import os

print("Training file downloaded:", os.path.exists("adult_ch6_training.csv"))
print("Test file downloaded:", os.path.exists("adult_ch6_test.csv"))


Training file downloaded: True
Test file downloaded: True


### Load the CSV files as Spark DataFrames

`spark.read.csv()` reads a CSV file into a Spark DataFrame.

Arguments:

- `header=True`: use the first row as column names;
- `inferSchema=True`: allow Spark to detect numeric and text column types automatically.


In [5]:
df_training = spark.read.csv(
    "adult_ch6_training.csv",
    header=True,
    inferSchema=True
)

df_testing = spark.read.csv(
    "adult_ch6_test.csv",
    header=True,
    inferSchema=True
)


### Preview the data

`show(5)` displays the first five rows without moving the whole dataset into Pandas.

In [6]:
df_training.show(5)

+---+--------------+------+----------------+
|age|Marital status|Income|Cap_Gains_Losses|
+---+--------------+------+----------------+
| 52| Never-married| <=50K|         0.02174|
| 37|      Divorced| <=50K|             0.0|
| 53|       Married| <=50K|             0.0|
| 45|       Married| <=50K|             0.0|
| 39|       Married| <=50K|             0.0|
+---+--------------+------+----------------+
only showing top 5 rows


### Check the column names and data types

`printSchema()` helps us confirm that:

- `age` and `Cap_Gains_Losses` are numeric;
- `Marital status` and `Income` are text columns.

In [7]:
df_training.printSchema()

root
 |-- age: integer (nullable = true)
 |-- Marital status: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- Cap_Gains_Losses: double (nullable = true)



## 4. Handle missing values

For this introductory exercise, we remove rows containing missing values with `dropna()`.

This is a simple approach, but it can reduce the dataset. In a real project, we would first investigate how many values are missing and why.

In [8]:
df_training = df_training.dropna()
df_testing = df_testing.dropna()

print("Training rows:", df_training.count())
print("Testing rows:", df_testing.count())

Training rows: 18761
Testing rows: 6155


## 5. Convert categorical columns into numbers

Machine-learning algorithms require numeric inputs.

`StringIndexer` assigns a numeric index to each category.

For example, `Income` may be converted to:

- one income category → `0.0`
- the other income category → `1.0`

`handleInvalid="keep"` prevents an error if the test dataset contains a category not seen during training.

### Why fit only on the training data?

The encoding rules must be learned from the training data and then applied unchanged to the test data. This avoids **data leakage** and keeps category mappings consistent.

In [9]:
indexers = [
    StringIndexer(
        inputCol="Marital status",
        outputCol="Marital statusNumeric",
        handleInvalid="keep"
    ),
    StringIndexer(
        inputCol="Income",
        outputCol="IncomeNumeric"
    )
]

encoding_pipeline = Pipeline(stages=indexers)

encoding_model = encoding_pipeline.fit(df_training)

df_training_encoded = encoding_model.transform(df_training)
df_testing_encoded = encoding_model.transform(df_testing)

### Inspect the encoded columns

The new columns are:

- `Marital statusNumeric`
- `IncomeNumeric`

In [10]:
df_training_encoded.select(
    "Marital status",
    "Marital statusNumeric",
    "Income",
    "IncomeNumeric"
).show(10)

+--------------+---------------------+------+-------------+
|Marital status|Marital statusNumeric|Income|IncomeNumeric|
+--------------+---------------------+------+-------------+
| Never-married|                  1.0| <=50K|          0.0|
|      Divorced|                  2.0| <=50K|          0.0|
|       Married|                  0.0| <=50K|          0.0|
|       Married|                  0.0| <=50K|          0.0|
|       Married|                  0.0| <=50K|          0.0|
|       Married|                  0.0|  >50K|          1.0|
| Never-married|                  1.0|  >50K|          1.0|
|       Married|                  0.0|  >50K|          1.0|
|       Married|                  0.0|  >50K|          1.0|
| Never-married|                  1.0| <=50K|          0.0|
+--------------+---------------------+------+-------------+
only showing top 10 rows


### Check the category-to-number mapping

This is important when interpreting the confusion matrix. The mapping can depend on category frequency, so do not assume which class is `0.0` or `1.0`.

In [11]:
for stage in encoding_model.stages:
    print(stage.getOutputCol(), "mapping:", list(enumerate(stage.labels)))

Marital statusNumeric mapping: [(0, 'Married'), (1, 'Never-married'), (2, 'Divorced'), (3, 'Separated'), (4, 'Widowed')]
IncomeNumeric mapping: [(0, '<=50K'), (1, '>50K')]


# Model 1: Marital Status + Capital Gains and Losses

## 6. Assemble the Model 1 input features

Spark ML expects all input variables inside one vector column named `features`.

`VectorAssembler` combines:

- `Marital statusNumeric`
- `Cap_Gains_Losses`

into a single vector.

In [12]:
assembler1 = VectorAssembler(
    inputCols=["Marital statusNumeric", "Cap_Gains_Losses"],
    outputCol="features"
)

model1_training = assembler1.transform(df_training_encoded)
model1_testing = assembler1.transform(df_testing_encoded)

We keep only the two columns needed by the classifier:

- `features`: input variables;
- `IncomeNumeric`: target or label.

In [13]:
model1_training = model1_training.select("features", "IncomeNumeric")
model1_testing = model1_testing.select("features", "IncomeNumeric")

model1_training.show(5, truncate=False)

+-------------+-------------+
|features     |IncomeNumeric|
+-------------+-------------+
|[1.0,0.02174]|0.0          |
|[2.0,0.0]    |0.0          |
|(2,[],[])    |0.0          |
|(2,[],[])    |0.0          |
|(2,[],[])    |0.0          |
+-------------+-------------+
only showing top 5 rows


## 7. Train Decision Tree Model 1

Arguments:

- `featuresCol="features"`: identifies the input vector;
- `labelCol="IncomeNumeric"`: identifies the correct target value;
- `seed=42`: supports reproducible results where randomness is involved.

`fit()` learns the decision rules from the training data.

In [14]:
dtc1 = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="IncomeNumeric",
    seed=42
)

model1 = dtc1.fit(model1_training)

## 8. Predict the test data

`transform()` applies the trained model to each test row.

Important output columns include:

- `prediction`: predicted class;
- `probability`: estimated probability for each class;
- `rawPrediction`: unnormalised confidence scores.

In [15]:
predictions1 = model1.transform(model1_testing)

predictions1.select(
    "IncomeNumeric",
    "prediction",
    "probability"
).show(10, truncate=False)

+-------------+----------+-----------------------------------------+
|IncomeNumeric|prediction|probability                              |
+-------------+----------+-----------------------------------------+
|0.0          |0.0       |[0.6295954671234681,0.3704045328765318]  |
|1.0          |1.0       |[0.18188914910226386,0.8181108508977362] |
|0.0          |0.0       |[0.9547733333333334,0.045226666666666665]|
|1.0          |0.0       |[0.9547733333333334,0.045226666666666665]|
|1.0          |0.0       |[0.6295954671234681,0.3704045328765318]  |
|0.0          |0.0       |[0.6295954671234681,0.3704045328765318]  |
|0.0          |1.0       |[0.18188914910226386,0.8181108508977362] |
|0.0          |0.0       |[0.9547733333333334,0.045226666666666665]|
|0.0          |0.0       |[0.6295954671234681,0.3704045328765318]  |
|0.0          |0.0       |[0.9547733333333334,0.045226666666666665]|
+-------------+----------+-----------------------------------------+
only showing top 10 rows


### View the decision tree rules

`toDebugString` provides a text representation of the decision tree.

- `feature 0` refers to `Marital statusNumeric`;
- `feature 1` refers to `Cap_Gains_Losses`.

In [16]:
print(model1.toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_9dd965657fff, depth=4, numNodes=11, numClasses=2, numFeatures=2
  If (feature 0 in {1.0,2.0,3.0,4.0})
   If (feature 1 <= 0.0497355)
    Predict: 0.0
   Else (feature 1 > 0.0497355)
    If (feature 1 <= 0.150222)
     If (feature 0 in {3.0})
      Predict: 0.0
     Else (feature 0 not in {3.0})
      Predict: 1.0
    Else (feature 1 > 0.150222)
     Predict: 0.0
  Else (feature 0 not in {1.0,2.0,3.0,4.0})
   If (feature 1 <= 0.0497355)
    Predict: 0.0
   Else (feature 1 > 0.0497355)
    Predict: 1.0



## 9. Model 1 contingency table

A contingency table compares **actual** and **predicted** classes.

- Rows are actual `IncomeNumeric` values.
- Columns are predicted values.
- Diagonal cells are correct predictions.
- Off-diagonal cells are errors.

`pivot("prediction")` turns prediction values into table columns.

In [17]:
contingency_table_1 = (
    predictions1
    .groupBy("IncomeNumeric")
    .pivot("prediction")
    .count()
    .fillna(0)
    .orderBy("IncomeNumeric")
)

print("Model 1 contingency table:")
contingency_table_1.show()

Model 1 contingency table:
+-------------+----+---+
|IncomeNumeric| 0.0|1.0|
+-------------+----+---+
|          0.0|4605| 69|
|          1.0|1106|375|
+-------------+----+---+



## 10. Calculate Model 1 evaluation metrics

### Accuracy

The proportion of all predictions that are correct.

### Precision

Of the cases predicted as a class, how many were correct?

### Recall

Of the actual cases in a class, how many did the model identify?

### F1 score

The harmonic mean of precision and recall. It is useful when both false positives and false negatives matter.

Spark's `weightedPrecision` and `weightedRecall` calculate averages weighted by the number of cases in each class.

In [18]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="IncomeNumeric",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="IncomeNumeric",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="IncomeNumeric",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="IncomeNumeric",
    predictionCol="prediction",
    metricName="f1"
)

accuracy1 = accuracy_evaluator.evaluate(predictions1)
precision1 = precision_evaluator.evaluate(predictions1)
recall1 = recall_evaluator.evaluate(predictions1)
f1_1 = f1_evaluator.evaluate(predictions1)

print("Model 1 Accuracy :", round(accuracy1, 4))
print("Model 1 Precision:", round(precision1, 4))
print("Model 1 Recall   :", round(recall1, 4))
print("Model 1 F1 score :", round(f1_1, 4))

Model 1 Accuracy : 0.8091
Model 1 Precision: 0.8155
Model 1 Recall   : 0.8091
Model 1 F1 score : 0.7672


### Optional: Area Under the ROC Curve

`BinaryClassificationEvaluator` evaluates a binary classifier.

- `rawPredictionCol="rawPrediction"` uses the model's confidence scores;
- `labelCol="IncomeNumeric"` identifies the true class;
- `metricName="areaUnderROC"` requests ROC-AUC.

ROC-AUC ranges from about `0.5` for random discrimination to `1.0` for perfect discrimination.

In [19]:
auc_evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction",
    labelCol="IncomeNumeric",
    metricName="areaUnderROC"
)

auc1 = auc_evaluator.evaluate(predictions1)
print("Model 1 Area Under ROC:", round(auc1, 4))

Model 1 Area Under ROC: 0.7113


### Optional detailed classification report

The Scikit-learn report shows class-specific precision, recall, and F1 scores.

`toPandas()` is appropriate here because we transfer only two small columns from the test predictions.

In [20]:
predictions1_pd = predictions1.select(
    "IncomeNumeric",
    "prediction"
).toPandas()

print(
    classification_report(
        predictions1_pd["IncomeNumeric"],
        predictions1_pd["prediction"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         0.0       0.81      0.99      0.89      4674
         1.0       0.84      0.25      0.39      1481

    accuracy                           0.81      6155
   macro avg       0.83      0.62      0.64      6155
weighted avg       0.82      0.81      0.77      6155



# Model 2: Marital Status + Age

Model 2 follows the same steps, but replaces `Cap_Gains_Losses` with `age`.

The learning activity specifically asks Model 2 to use:

- `Marital status`
- `age`

## 11. Assemble the Model 2 features

In [21]:
assembler2 = VectorAssembler(
    inputCols=["Marital statusNumeric", "age"],
    outputCol="features"
)

model2_training = assembler2.transform(df_training_encoded)
model2_testing = assembler2.transform(df_testing_encoded)

model2_training = model2_training.select("features", "IncomeNumeric")
model2_testing = model2_testing.select("features", "IncomeNumeric")

## 12. Train Model 2 and predict the test data

In [22]:
dtc2 = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="IncomeNumeric",
    seed=42
)

model2 = dtc2.fit(model2_training)
predictions2 = model2.transform(model2_testing)

### View the Model 2 decision tree

For Model 2:

- `feature 0` is `Marital statusNumeric`;
- `feature 1` is `age`.

In [23]:
print(model2.toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_95230be2ba36, depth=2, numNodes=5, numClasses=2, numFeatures=2
  If (feature 1 <= 54.5)
   If (feature 1 <= 34.5)
    Predict: 1.0
   Else (feature 1 > 34.5)
    Predict: 0.0
  Else (feature 1 > 54.5)
   Predict: 1.0



## 13. Model 2 contingency table

In [24]:
contingency_table_2 = (
    predictions2
    .groupBy("IncomeNumeric")
    .pivot("prediction")
    .count()
    .fillna(0)
    .orderBy("IncomeNumeric")
)

print("Model 2 contingency table:")
contingency_table_2.show()

Model 2 contingency table:
+-------------+----+---+
|IncomeNumeric| 0.0|1.0|
+-------------+----+---+
|          0.0|4674|  0|
|          1.0| 591|890|
+-------------+----+---+



## 14. Calculate Model 2 metrics

We reuse the same evaluator objects so the two models are measured consistently.

In [25]:
accuracy2 = accuracy_evaluator.evaluate(predictions2)
precision2 = precision_evaluator.evaluate(predictions2)
recall2 = recall_evaluator.evaluate(predictions2)
f1_2 = f1_evaluator.evaluate(predictions2)
auc2 = auc_evaluator.evaluate(predictions2)

print("Model 2 Accuracy :", round(accuracy2, 4))
print("Model 2 Precision:", round(precision2, 4))
print("Model 2 Recall   :", round(recall2, 4))
print("Model 2 F1 score :", round(f1_2, 4))
print("Model 2 ROC-AUC  :", round(auc2, 4))

Model 2 Accuracy : 0.904
Model 2 Precision: 0.9148
Model 2 Recall   : 0.904
Model 2 F1 score : 0.8949
Model 2 ROC-AUC  : 0.6972


### Optional detailed classification report for Model 2

In [26]:
predictions2_pd = predictions2.select(
    "IncomeNumeric",
    "prediction"
).toPandas()

print(
    classification_report(
        predictions2_pd["IncomeNumeric"],
        predictions2_pd["prediction"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

         0.0       0.89      1.00      0.94      4674
         1.0       1.00      0.60      0.75      1481

    accuracy                           0.90      6155
   macro avg       0.94      0.80      0.85      6155
weighted avg       0.91      0.90      0.89      6155



# 15. Compare Model 1 and Model 2

A single comparison table makes it easier to identify which model performs better.

A higher value is generally better for all metrics shown below. However, the most important metric depends on the business problem and the relative cost of different errors.

In [27]:
model_comparison = pd.DataFrame({
    "Model": [
        "Model 1: Marital status + Capital gains/losses",
        "Model 2: Marital status + Age"
    ],
    "Accuracy": [accuracy1, accuracy2],
    "Precision": [precision1, precision2],
    "Recall": [recall1, recall2],
    "F1 score": [f1_1, f1_2],
    "ROC-AUC": [auc1, auc2]
})

model_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1 score,ROC-AUC
0,Model 1: Marital status + Capital gains/losses,0.8091,0.8155,0.8091,0.7672,0.7113
1,Model 2: Marital status + Age,0.9040,0.9148,0.9040,0.8949,0.6972


## Questions for interpreting the comparison

Consider the following before selecting the better model:

1. Which model has the highest accuracy?
2. Is the same model also best for precision, recall, and F1?
3. Does one model perform well on the majority class but poorly on the minority class?
4. Which types of error appear most often in each contingency table?
5. Is the performance difference large enough to matter in practice?
6. Which model is easier to explain to a non-technical audience?

# Additional beginner practice: contingency tables for input variables

The cells below are optional. They use Pandas `crosstab()` to examine how an input variable is distributed across income classes.

This is different from the model contingency table:

- **Model contingency table:** actual class versus predicted class.
- **Exploratory contingency table:** an input category versus the target class.

In [28]:
df_training_pd = df_training.toPandas()
df_training_pd.head()

,age,Marital status,Income,Cap_Gains_Losses
0,52,Never-married,<=50K,0.02174
1,37,Divorced,<=50K,0.00000
2,53,Married,<=50K,0.00000
3,45,Married,<=50K,0.00000
4,39,Married,<=50K,0.00000


## Marital status versus income

`pd.crosstab()` counts the number of records for each combination of categories.

In [29]:
marital_status_table = pd.crosstab(
    df_training_pd["Marital status"],
    df_training_pd["Income"]
)

marital_status_table

Income,<=50K,>50K
Marital status,,
Divorced,2292,266
Married,5011,3859
Never-married,5885,287
Separated,559,38
Widowed,524,40


## Convert age into groups

`pd.cut()` converts a continuous numeric variable into labelled intervals.

Arguments:

- `bins`: interval boundaries;
- `labels`: names assigned to those intervals.

In [ ]:
import numpy as np

bins = [0, 20, 30, 40, 50, 60, 70, 80, np.inf]
names = ["<20", "20-30", "30-40", "40-50",
         "50-60", "60-70", "70-80", "80+"]

df_training_pd["age_group"] = pd.cut(
    df_training_pd["age"],
    bins=bins,
    labels=names
)

In [ ]:
age_income_table = pd.crosstab(
    df_training_pd["age_group"],
    df_training_pd["Income"]
)

age_income_table

# Learning Activity 2: Why different problems need different metrics

The statement is correct: **different evaluation metrics are appropriate for different problems**.

| Scenario | Main concern | Useful metric |
|---|---|---|
| Fraud detection | Missing a fraudulent transaction | Recall |
| Spam filtering | Blocking a genuine email | Precision |
| Balanced general classification | Overall correct predictions | Accuracy |
| Imbalanced classification | Balance precision and recall | F1 score |
| Ranking customers by risk | Discrimination across thresholds | ROC-AUC or PR-AUC |
| Predicting a numeric value | Size of prediction errors | MAE, RMSE, or R² |
| Clustering without labels | Separation and cohesion of clusters | Silhouette score |

There is no single metric that is best for every situation. The metric should reflect the goal and the cost of errors.

## Suggested response for the discussion forum

Use your own words and include:

- which model performed better overall;
- the metric values that support your conclusion;
- whether the contingency tables reveal any important error pattern;
- whether accuracy alone would have produced the same conclusion;
- which metric you would prioritise in a realistic income-classification application; and
- one limitation of the evaluation.

### Peer feedback

When responding to another student:

1. identify one point you agree or disagree with;
2. refer to a metric or contingency-table result;
3. explain why that evidence supports your view; and
4. suggest one additional issue they could consider.

## 16. Stop Spark

Stopping Spark releases the resources used by the session.

Run this cell only after completing the notebook.

In [ ]:
spark.stop()